# experiment_rv

## Reproducibility bootstrap
Run first. Resolves paths for the authors' Drive, a fresh Colab (clones the anon repo), or a local clone. No edits needed.

In [ ]:
# === Reproducibility bootstrap (public bundle) ===
# Resolves all paths for: (a) authors' Google Drive, (b) fresh Colab (clones repo),
# (c) local clone. Sets CODE_DIR, DATA_DIR, RESULTS_DIR, DATA_PATH. Run first; no edits needed.
import os, sys
from pathlib import Path

RESULTS_SUBFOLDER = "experiment_rv"
DATA_FILENAME = "rv_dataset.csv"   # None for synthetic experiments

def _resolve():
    try:
        import google.colab  # noqa: F401
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        dr = Path('/content/drive/MyDrive')
        if (dr/'GNAVAR'/'code'/'gnavar_core.py').exists():
            b = dr/'GNAVAR'; return b/'code', b/'data', b/'results'
        # Fresh Colab without the authors' Drive: the repo files must be present in the
        # session. Anonymous-review repos cannot be git-cloned, so upload the bundle:
        #   1) Download the ZIP from the Anonymous GitHub page (Download / ZIP button).
        #   2) In Colab, upload the ZIP via the Files pane, then in a cell run:
        #        !unzip -o your_bundle.zip
        #   3) %cd into the unzipped repo folder, then run this notebook.
        for cand in [Path('/content')/'ICDM-GNAVAR-EDAE', Path.cwd()]:
            if (cand/'src'/'gnavar_core.py').exists():
                return cand/'src', cand/'data', cand/'results'
        raise FileNotFoundError(
            'Repo files not found in the Colab session. Download the ZIP from the '
            'Anonymous GitHub page, upload and unzip it here, then %cd into the folder '
            'and re-run. See the repository README, Path B, Option B1.')
    except ImportError:
        repo = Path.cwd()
        while repo != repo.parent and not (repo/'verify_paper_numbers.py').exists():
            repo = repo.parent
        return repo/'src', repo/'data', repo/'results'

CODE_DIR, DATA_DIR, RESULTS_ROOT = _resolve()
sys.path.insert(0, str(CODE_DIR))
RESULTS_DIR = RESULTS_ROOT / RESULTS_SUBFOLDER
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = (DATA_DIR / DATA_FILENAME) if DATA_FILENAME else None
DRIVE_ROOT = str(CODE_DIR.parent.parent)  # back-compat for any cell referencing DRIVE_ROOT
print('CODE_DIR    =', CODE_DIR)
print('RESULTS_DIR =', RESULTS_DIR)
if DATA_PATH: print('DATA_PATH   =', DATA_PATH)
import numpy as np
import pandas as pd
import json, hashlib, datetime, time, itertools, platform
try:
    import torch
    import torch.nn as nn
except Exception:
    pass
from gnavar_core import *  # model, generator, fit/eval utils
# === end bootstrap ===


# Realized Volatility: Diagnostic Validation (Negative Control)

**Purpose.** Test whether the effective-rank diagnostic correctly predicts a real-data case where modulator recovery is *not* feasible. This is the contrasting case to Beijing: where Beijing had high effective rank and clean recovery, the RV panel has collapsed support, and we predict (in advance) that recovery will be unreliable.

## The dataset

Daily realized volatility for 8 international equity indices (S&P 500, DAX, CAC 40, FTSE 100, OMX Stockholm, Nikkei 225, KOSPI, Hang Seng), 2,615 trading days. Realized volatility is famously persistent (AR(1) ~ 0.8) and cross-correlated (European indices correlate up to 0.94). Both properties collapse the joint lag-block support.

## The a-priori prediction (committed before fitting)

Standard finance literature treats the S&P 500 as the global volatility leader, so a natural hypothesis would be: **SPX is the dominant modulator of within-region volatility spillover.** On Beijing this kind of hypothesis was confirmed (TEMP modulated NO2 -> O3 in 4/4 sites).

**But the effective-rank diagnostic predicts this recovery will fail here.** Pre-fit, the joint source lag-block effective rank for each target is below 2.0 (versus the synthetic identifiability cliff at ~1.6, and versus Beijing Setup A at >4.0). Corollary 8.2 says gates become non-identifiable as joint support collapses. So we predict:

1. SPX will **not** consistently rank #1 as a modulator across targets.
2. The identity of the top modulator will **vary** target-to-target (no cross-target consistency).
3. Top-modulator margins over second place will be **small** (~1x), versus Beijing's 2-59x.
4. Two independent seeds will **disagree** on the top modulator (within-target instability).

If these hold, the diagnostic has correctly flagged a real-world case where interaction discovery is infeasible -- demonstrating discriminating power, not just rubber-stamping.

## Design

- **4 targets**: FTSE, GDAXI (European, most collapsed), N225, HSI (Asian, slightly higher r_eff)
- Per target: sources = regional peers + SPX (the modulator candidate)
- **log-transform then z-score** each series (standard RV practice; raw RV has skew 3-4, log-RV ~0.5)
- One continuous series (no run gaps); standard make_lag_tensor, K=2
- Chronological 80/20 train/test split
- **Two independent G-NAVAR seeds per target** for within-target stability check
- Pairwise NAVAR baseline for MSE comparison

## Outputs (under `/content/drive/MyDrive/GNAVAR/results/experiment_rv/`)

- `results.csv`: per-target metrics, both seeds
- `prediction_check.txt`: explicit verdict on each of the 4 committed predictions
- `metadata.json`

## Cell 1: Drive mount and data file

## Cell 2: Imports

In [ ]:
from gnavar_core import *
import time, json, hashlib, platform, datetime
from dataclasses import asdict
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

print(f'Device: {DEVICE} | Mixed precision: {USE_AMP}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

## Cell 3: Targets and config

In [ ]:
INDICES = ['.SPX', '.GDAXI', '.FCHI', '.FTSE', '.OMXSPI', '.N225', '.KS11', '.HSI']

# Each target: regional peers + SPX as the modulator candidate.
# The a-priori hypothesis is that SPX dominates as a modulator of the
# peer-edges. The diagnostic predicts this will FAIL (low r_eff).
TARGETS = {
    'FTSE':  {'target': '.FTSE',  'sources': ['.GDAXI', '.FCHI', '.OMXSPI', '.SPX']},
    'GDAXI': {'target': '.GDAXI', 'sources': ['.FCHI', '.FTSE', '.OMXSPI', '.SPX']},
    'N225':  {'target': '.N225',  'sources': ['.KS11', '.HSI', '.FTSE', '.SPX']},
    'HSI':   {'target': '.HSI',   'sources': ['.KS11', '.N225', '.FTSE', '.SPX']},
}
MODULATOR_CANDIDATE = '.SPX'  # the hypothesized dominant modulator

K = 2
BASE_SEED = 42
N_RESTARTS = 3
N_STABILITY_SEEDS = 2   # two independent fits per target for stability check
TEST_FRACTION = 0.20

def cfg_for_target():
    return Config(n_vars=5, K=K, hidden_dim=32, n_epochs=300,
                  l1_lambda=0.005, triviality_threshold=0.001)

print(f'Targets: {list(TARGETS.keys())}')
print(f'Modulator candidate (hypothesized dominant): {MODULATOR_CANDIDATE}')
print(f'Stability seeds per target: {N_STABILITY_SEEDS}')
print(f'Total fits: {len(TARGETS) * N_STABILITY_SEEDS * N_RESTARTS} G-NAVAR + '
      f'{len(TARGETS) * N_RESTARTS} Pairwise')

## Cell 4: Load, log-transform, z-score

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
X_raw = df_raw[INDICES].values
print(f'Raw RV shape: {X_raw.shape}')

# Log-transform (RV is log-normal-ish; raw skew 3-4, log skew ~0.5)
assert (X_raw > 0).all(), 'RV must be positive for log transform'
X_log = np.log(X_raw)

# Z-score per index
means = X_log.mean(axis=0)
stds  = X_log.std(axis=0)
X_z = ((X_log - means) / stds).astype(np.float32)

from scipy import stats
print('Per-index: log-RV skew (should be ~0.5-1.0) and AR(1):')
for i, name in enumerate(INDICES):
    sk = stats.skew(X_log[:, i])
    ar1 = np.corrcoef(X_z[:-1, i], X_z[1:, i])[0, 1]
    print(f'  {name:<9s} skew={sk:+.2f}  AR(1)={ar1:.3f}')

idx_of = {name: i for i, name in enumerate(INDICES)}

def build_target_data(target_name):
    spec = TARGETS[target_name]
    cols = [idx_of[spec['target']]] + [idx_of[s] for s in spec['sources']]
    return X_z[:, cols], spec['sources']  # target at col 0, source names in order

## Cell 5: Trial driver (per target, with stability seeds)

In [ ]:
@torch.no_grad()
def top_modulator_per_edge(model, X_lag_t, source_names, threshold):
    """For each source edge, return (top_modulator_name, top_score, ranked_list)."""
    n = len(source_names)
    out = {}
    for j in range(n):
        scores = []
        for k in range(n):
            if k == j: continue
            s = gate_triviality_score(model, X_lag_t, j, k)
            scores.append((source_names[k], s))
        scores.sort(key=lambda x: -x[1])
        out[source_names[j]] = scores
    return out

def run_rv_target(target_name, data_z, cfg):
    """Fit G-NAVAR (N_STABILITY_SEEDS independent) + Pairwise on one target."""
    t0 = time.time()
    X_sub, source_names = build_target_data(target_name)
    n_total = X_sub.shape[0]
    n_train = int(n_total * (1 - TEST_FRACTION))
    X_train_traj = X_sub[:n_train]
    X_test_traj  = X_sub[n_train:]

    X_lag_train, y_train = make_lag_tensor(X_train_traj, K=cfg.K)
    X_lag_test,  y_test  = make_lag_tensor(X_test_traj,  K=cfg.K)
    r_eff = effective_rank_full(X_lag_train)
    X_lag_t = torch.from_numpy(X_lag_train).to(DEVICE)

    # Fit N_STABILITY_SEEDS independent G-NAVAR models
    seed_results = []
    for s in range(N_STABILITY_SEEDS):
        seed = BASE_SEED + 1000 * s
        m = fit_gnavar_from_lag_with_restarts(X_lag_train, y_train, cfg,
                                              seed=seed, n_restarts=N_RESTARTS, verbose=False)
        m.eval()
        mse_test = held_out_mse_gnavar_from_lag(m, X_lag_test, y_test)
        edges = top_modulator_per_edge(m, X_lag_t, source_names, cfg.triviality_threshold)
        seed_results.append({'seed': seed, 'mse_test': mse_test, 'edges': edges})

    # Pairwise baseline (single fit)
    m_pw = fit_pairwise_from_lag_with_restarts(X_lag_train, y_train, cfg,
                                               seed=BASE_SEED, n_restarts=N_RESTARTS, verbose=False)
    m_pw.eval()
    mse_pw_test = held_out_mse_pairwise_from_lag(m_pw, X_lag_test, y_test)

    # --- Analyze the committed predictions ---
    # Among the non-SPX peer edges, how often does SPX rank #1 as modulator?
    peer_sources = [s for s in source_names if s != MODULATOR_CANDIDATE]
    # Use seed 0 for the primary modulator ranking
    edges0 = seed_results[0]['edges']
    spx_rank1_count = 0
    margins = []
    for peer in peer_sources:
        ranked = edges0[peer]  # list of (modulator_name, score)
        rank_names = [name for name, _ in ranked]
        if rank_names and rank_names[0] == MODULATOR_CANDIDATE:
            spx_rank1_count += 1
        # margin top/2nd
        if len(ranked) >= 2 and ranked[1][1] > 1e-12:
            margins.append(ranked[0][1] / ranked[1][1])
        else:
            margins.append(float('inf'))

    # Top modulator overall (across all edges, seed 0): most frequent rank-1
    rank1_mods = [edges0[src][0][0] for src in source_names if edges0[src]]
    from collections import Counter
    top_mod_counter = Counter(rank1_mods)
    most_common_top_mod, most_common_count = top_mod_counter.most_common(1)[0]

    # Seed stability: do seed 0 and seed 1 agree on the top modulator per edge?
    if N_STABILITY_SEEDS >= 2:
        edges1 = seed_results[1]['edges']
        agree = sum(1 for src in source_names
                    if edges0[src] and edges1[src] and edges0[src][0][0] == edges1[src][0][0])
        seed_agreement = agree / len(source_names)
    else:
        seed_agreement = float('nan')

    mean_margin = float(np.mean([m for m in margins if m != float('inf')])) if margins else float('nan')

    return {
        'target': target_name,
        'r_eff': r_eff,
        'n_train': X_lag_train.shape[0],
        'n_test': X_lag_test.shape[0],
        'mse_test_gnavar_seed0': seed_results[0]['mse_test'],
        'mse_test_gnavar_seed1': seed_results[1]['mse_test'] if N_STABILITY_SEEDS >= 2 else float('nan'),
        'mse_test_pairwise': mse_pw_test,
        'mse_ratio_pw_to_gn': mse_pw_test / seed_results[0]['mse_test'],
        'spx_rank1_count': spx_rank1_count,
        'n_peer_edges': len(peer_sources),
        'mean_top_margin': mean_margin,
        'most_common_top_modulator': most_common_top_mod,
        'most_common_top_count': most_common_count,
        'seed_agreement': seed_agreement,
        'edges_seed0_json': json.dumps({k: v for k, v in edges0.items()}),
        'elapsed_seconds': time.time() - t0,
    }

## Cell 6: Resume-aware driver

In [ ]:
RESULTS_CSV = RESULTS_DIR / 'results.csv'

def load_existing():
    if RESULTS_CSV.exists():
        d = pd.read_csv(RESULTS_CSV)
        return d, set(d['target'])
    return pd.DataFrame(), set()

def append_result(row):
    write_header = not RESULTS_CSV.exists()
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=write_header, index=False)

def run_rv_sweep():
    _, completed = load_existing()
    print(f'Already completed: {len(completed)} targets')
    todo = [t for t in TARGETS if t not in completed]
    print(f'To run: {len(todo)} targets')
    cfg = cfg_for_target()
    for i, target_name in enumerate(todo, 1):
        print(f'\n[{i}/{len(todo)}] target={target_name}', flush=True)
        result = run_rv_target(target_name, X_z, cfg)
        append_result(result)
        print(f'  {result["elapsed_seconds"]:.1f}s | r_eff={result["r_eff"]:.3f} | '
              f'SPX rank-1 in {result["spx_rank1_count"]}/{result["n_peer_edges"]} edges | '
              f'mean margin={result["mean_top_margin"]:.2f}x | '
              f'seed agreement={result["seed_agreement"]:.2f} | '
              f'top mod={result["most_common_top_modulator"]}', flush=True)
    return load_existing()[0]

## Cell 7: Reproducibility metadata

In [ ]:
def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

import gnavar_core as _gc
md = {
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'experiment': 'rv_diagnostic_validation',
    'data_file_sha256': _sha256(DATA_PATH),
    'gnavar_core_sha256': _sha256(_gc.__file__),
    'targets': {k: v for k, v in TARGETS.items()},
    'modulator_candidate': MODULATOR_CANDIDATE,
    'K': K, 'n_restarts': N_RESTARTS, 'n_stability_seeds': N_STABILITY_SEEDS,
    'test_fraction': TEST_FRACTION,
    'preprocessing': 'log-transform then z-score per index',
    'training_config': asdict(cfg_for_target()),
    'python_version': platform.python_version(),
    'torch_version': torch.__version__,
    'numpy_version': np.__version__,
    'device': str(DEVICE), 'use_amp': USE_AMP,
    'cuda_device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
(RESULTS_DIR / 'metadata.json').write_text(json.dumps(md, indent=2))
print(json.dumps(md, indent=2))

## Cell 8: Run the sweep

RV is a short series (2,615 days), so each fit is fast. Total ~5-10 min.

In [ ]:
df = run_rv_sweep()
print(f'\nDone. Total rows: {len(df)}')

## Cell 9: Verdict on the committed predictions

In [ ]:
def check_predictions(df):
    lines = [
        'Realized Volatility: diagnostic-validation verdict',
        '=' * 70,
        'Pre-committed prediction: r_eff < 2 for all targets implies modulator',
        'recovery will be UNRELIABLE. We test 4 specific consequences.',
        '',
        'Effective rank per target (all predicted < 2.0):',
    ]
    for _, r in df.iterrows():
        flag = 'below cliff' if r['r_eff'] < 1.6 else ('marginal' if r['r_eff'] < 2.0 else 'ABOVE 2.0')
        lines.append(f"  {r['target']:<8s} r_eff = {r['r_eff']:.3f}  ({flag})")
    lines.append('')

    # Prediction 1: SPX does not consistently rank #1
    lines.append('PREDICTION 1: SPX does not consistently dominate as modulator.')
    total_edges = df['n_peer_edges'].sum()
    total_spx1  = df['spx_rank1_count'].sum()
    for _, r in df.iterrows():
        lines.append(f"  {r['target']:<8s} SPX ranked #1 in {r['spx_rank1_count']}/{r['n_peer_edges']} peer edges")
    lines.append(f"  TOTAL: SPX #1 in {total_spx1}/{total_edges} peer edges across all targets")
    p1_holds = (total_spx1 / total_edges) < 0.5 if total_edges else None
    lines.append(f"  => Prediction {'HOLDS' if p1_holds else 'FAILS'}: SPX is "
                 f"{'not' if p1_holds else ''} the consistent dominant modulator")
    lines.append('')

    # Prediction 2: top modulator varies across targets
    lines.append('PREDICTION 2: the top modulator varies target-to-target.')
    top_mods = df.set_index('target')['most_common_top_modulator'].to_dict()
    for t, m in top_mods.items():
        lines.append(f"  {t:<8s} most common top modulator: {m}")
    n_distinct = df['most_common_top_modulator'].nunique()
    p2_holds = n_distinct > 1
    lines.append(f"  Distinct top modulators across {len(df)} targets: {n_distinct}")
    lines.append(f"  => Prediction {'HOLDS' if p2_holds else 'FAILS'}: top modulator "
                 f"{'varies' if p2_holds else 'is consistent'} across targets")
    lines.append('')

    # Prediction 3: small margins
    lines.append('PREDICTION 3: top-modulator margins are small (~1x), vs Beijing 2-59x.')
    for _, r in df.iterrows():
        lines.append(f"  {r['target']:<8s} mean top-modulator margin: {r['mean_top_margin']:.2f}x")
    mean_margin_all = df['mean_top_margin'].mean()
    p3_holds = mean_margin_all < 2.0
    lines.append(f"  Mean margin across targets: {mean_margin_all:.2f}x")
    lines.append(f"  => Prediction {'HOLDS' if p3_holds else 'FAILS'}: margins are "
                 f"{'small' if p3_holds else 'large'} (Beijing Setup A was 2.0-59.3x)")
    lines.append('')

    # Prediction 4: seeds disagree
    lines.append('PREDICTION 4: independent seeds disagree on top modulator (instability).')
    for _, r in df.iterrows():
        lines.append(f"  {r['target']:<8s} seed agreement on top modulator: {r['seed_agreement']:.2f}")
    mean_agreement = df['seed_agreement'].mean()
    p4_holds = mean_agreement < 1.0
    lines.append(f"  Mean seed agreement: {mean_agreement:.2f} (1.0 = perfect agreement)")
    lines.append(f"  => Prediction {'HOLDS' if p4_holds else 'FAILS'}: seeds "
                 f"{'disagree' if p4_holds else 'agree'} (instability "
                 f"{'present' if p4_holds else 'absent'})")
    lines.append('')

    # MSE comparison
    lines.append('Forecast MSE (G-NAVAR seed0 vs Pairwise):')
    gn_wins = 0
    for _, r in df.iterrows():
        if r['mse_test_gnavar_seed0'] < r['mse_test_pairwise']:
            gn_wins += 1
        lines.append(f"  {r['target']:<8s} Gn={r['mse_test_gnavar_seed0']:.4f}  "
                     f"Pw={r['mse_test_pairwise']:.4f}  ratio={r['mse_ratio_pw_to_gn']:.2f}x")
    lines.append(f"  G-NAVAR wins on MSE: {gn_wins}/{len(df)}")
    lines.append('')

    # Overall verdict
    n_hold = sum([bool(p1_holds), bool(p2_holds), bool(p3_holds), bool(p4_holds)])
    lines.append('=' * 70)
    lines.append(f'OVERALL: {n_hold}/4 predictions hold.')
    if n_hold >= 3:
        lines.append('The diagnostic CORRECTLY flagged RV as a case where modulator')
        lines.append('recovery is unreliable. This validates its discriminating power:')
        lines.append('high r_eff (Beijing) -> clean recovery; low r_eff (RV) -> unreliable.')
    elif n_hold <= 1:
        lines.append('Unexpectedly, recovery looks more stable than the low r_eff predicted.')
        lines.append('This would WEAKEN the diagnostic claim and needs investigation.')
    else:
        lines.append('Mixed: some instability signatures present, others not. Report as mixed.')

    summary = '\n'.join(lines)
    (RESULTS_DIR / 'prediction_check.txt').write_text(summary)
    print(summary)

check_predictions(df)